Supplementary Code S8
Revised architectures under leakage-aware feature governance and nested cross-validation
Purpose

This notebook implements a revised internal validation framework for AI-based cardiac biomarker modeling after myocardial infarction, addressing preprocessing-induced information loss and feature-governance issues identified during earlier exploratory analyses.
The analysis is explicitly methodological and diagnostic: it evaluates model performance under corrected feature preprocessing and strict nested cross-validation, without introducing any external or locked test data.

The primary goal of S8 is to establish a clean, leakage-aware internal benchmarking baseline that can be reliably extended in subsequent validation stages (S9 and S10).

Relationship to prior scripts

S2 defines the original nested cross-validation protocol and identifies preliminary winner models across feature-set variants.

S5–S6 summarize performance–complexity trade-offs and audit potential leakage and dominant predictors under the original preprocessing assumptions.

S7 performs a stepwise ablation analysis under fixed S2 configurations to assess driver dependence and robustness.

S8 revises the feature preprocessing and governance layer, correcting preprocessing artifacts (e.g., silent feature loss due to numeric coercion), and re-evaluates revised model architectures using nested cross-validation.

Importantly, S8 does not introduce a locked test set and does not attempt to estimate deployment-level generalization.

Key principles

Uses the same target definition, cohort filtering, and outcome encoding as in S2–S7.

Feature sets are defined exclusively by externally governed lists (features_used_FULL.csv, features_used_CLINICAL.csv, features_used_BIOMARKERS.csv) to ensure cross-script consistency.

Binary and categorical clinical variables encoded as text (e.g., yes/no, true/false, tak/nie) are explicitly recoded into numeric form prior to numeric coercion.

All-missing and constant predictors are removed transparently and logged, preventing silent preprocessing artifacts.

Model selection and performance estimation are strictly separated using nested cross-validation.

No external data or locked test set is used at this stage.

Methods implemented in S8

For each feature-set variant (FULL, CLINICAL, BIOMARKERS), the notebook performs:

Leakage-aware feature preprocessing and governance

Input features are filtered to those present in the governed feature lists.
Binary textual predictors are recoded into numeric values before numeric coercion.
Predictors that are entirely missing or constant after preprocessing are removed and logged per variant.

Variant-specific matrix construction

Three predefined feature variants are evaluated:

FULL: combined clinical and biomarker features,

CLINICAL: conservative clinical features only,

BIOMARKERS: laboratory and molecular biomarkers only.

Each variant is processed independently, yielding an auditable feature matrix and diagnostic summary.

Nested cross-validation under revised preprocessing

For each variant, revised model architectures are evaluated using nested cross-validation:

inner folds are used exclusively for hyperparameter tuning,

outer folds are used exclusively for performance estimation.

This design yields unbiased out-of-fold estimates while preventing information leakage from tuning into evaluation.

Revised model architectures

The following classifiers are evaluated under identical validation conditions:

L1-regularized logistic regression,

elastic-net logistic regression,

support vector machines,

shallow random forests.

Optional recursive feature elimination (RFE) can be enabled as an experimental option; when used, feature selection is nested entirely within training folds.

Inputs (from S2–S7 workflow)

Required:

processed_dataset.csv

features_used_FULL.csv

features_used_CLINICAL.csv

features_used_BIOMARKERS.csv

Optional:

audit_nonnumeric_features.csv

All inputs are identical to, or derived from, those used in earlier stages, ensuring full pipeline continuity.

Outputs

Per-variant and aggregated tables:

S8_variant_matrix_diagnostics.csv — diagnostic summary of feature governance, including recoded binaries, dropped predictors, and final matrix dimensionality.

S8_feature_reduction_summary.csv — consolidated overview of feature reduction per variant.

S8_nestedcv_results_revised.csv — nested cross-validation performance metrics for all model–variant combinations.

S8_revised_winner_by_variant.csv — best-performing model per variant based on mean outer-fold ROC AUC.

S8_run_log.json — fully auditable record of inputs, configuration, preprocessing decisions, and outputs.

Interpretation note

Supplementary Code S8 establishes a corrected and methodologically robust internal validation baseline.
The reported performance estimates reflect improved feature handling rather than increased model complexity and should be interpreted as internal benchmarks only.
No claims regarding external validity or clinical readiness are made at this stage; such assessments are deferred to the locked-test evaluation (S9) and permutation-based robustness analysis (S10).

## Setup + catalogs

In [1]:
from __future__ import annotations

import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, matthews_corrcoef, brier_score_loss

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*Skipping features without any observed values.*")

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()

OUT_DIR = ROOT / "results" / "S8"
TAB_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"
FIG_DIR = OUT_DIR / "figures"

for d in (TAB_DIR, LOG_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Writing outputs to:", OUT_DIR.resolve())


ROOT: /content
Writing outputs to: /content/results/S8


## Finder + loadery

In [2]:
SEARCH_ROOTS = [
    ROOT,
    ROOT / "results",
    ROOT / "results" / "S2_nestedcv",
    ROOT / "results" / "S6_leakage_audit",
    ROOT / "results" / "S8_revised_models",
    ROOT / "results" / "S8_revised_nestedcv",
    Path("/mnt/data"),
    ROOT / "drive",
    ROOT / "drive" / "MyDrive",
]

def find_file(filename: str, roots: list[Path]) -> Path:
    for r in roots:
        p = r / filename
        if p.exists():
            return p
    for r in roots:
        if r.exists():
            hits = list(r.rglob(filename))
            if hits:
                hits = sorted(hits, key=lambda x: len(str(x)))
                return hits[0]
    raise FileNotFoundError(
        f"Missing required file: {filename}\nSearched in:\n - " + "\n - ".join(map(str, roots))
    )

def read_csv_required(name: str) -> tuple[pd.DataFrame, Path]:
    p = find_file(name, SEARCH_ROOTS)
    df = pd.read_csv(p)
    print(f"Loaded: {p} | shape={df.shape}")
    return df, p


## CV configuration

In [3]:
N_OUTER = 5
N_INNER = 5

USE_RFE = False
RFE_N_SELECT = 10

cv_outer = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
cv_inner = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)

print("S8 config:",
      f"OUTER={N_OUTER}, INNER={N_INNER}, USE_RFE={USE_RFE}, RFE_N_SELECT={RFE_N_SELECT}")


S8 config: OUTER=5, INNER=5, USE_RFE=False, RFE_N_SELECT=10


## Loading dataset + feature list

In [4]:
df_raw, path_processed = read_csv_required("processed_dataset.csv")

p_full = find_file("features_used_FULL.csv", SEARCH_ROOTS)
p_clin = find_file("features_used_CLINICAL.csv", SEARCH_ROOTS)
p_bio  = find_file("features_used_BIOMARKERS.csv", SEARCH_ROOTS)

def load_feature_list(path: Path) -> list[str]:
    fdf = pd.read_csv(path)
    fdf.columns = [c.strip() for c in fdf.columns]
    for col in ["feature", "features", "feature_name", "column", "variable", "var", "name"]:
        if col in fdf.columns:
            feats = fdf[col].astype(str).str.strip().tolist()
            return [f for f in feats if f and f.lower() != "nan"]
    feats = fdf.iloc[:, 0].astype(str).str.strip().tolist()
    return [f for f in feats if f and f.lower() != "nan"]

features_used = {
    "FULL": load_feature_list(p_full),
    "CLINICAL": load_feature_list(p_clin),
    "BIOMARKERS": load_feature_list(p_bio),
}

print("Governed feature counts:", {k: len(v) for k, v in features_used.items()})


pd.Series(features_used["FULL"]).to_csv(TAB_DIR / "S8__features_used_FULL_snapshot.csv", index=False, header=["feature"])
pd.Series(features_used["CLINICAL"]).to_csv(TAB_DIR / "S8__features_used_CLINICAL_snapshot.csv", index=False, header=["feature"])
pd.Series(features_used["BIOMARKERS"]).to_csv(TAB_DIR / "S8__features_used_BIOMARKERS_snapshot.csv", index=False, header=["feature"])


Loaded: /content/processed_dataset.csv | shape=(152, 117)
Governed feature counts: {'FULL': 69, 'CLINICAL': 58, 'BIOMARKERS': 10}


## Target

In [5]:
TARGET_COL = "typ_zawalu"
if TARGET_COL not in df_raw.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in processed_dataset.csv")

labels = df_raw[TARGET_COL].astype(str).str.strip()
allowed = {"STEMI", "NSTEMI"}

df = df_raw.loc[labels.isin(allowed)].copy()
y = (df[TARGET_COL].astype(str).str.strip() == "STEMI").astype(int).values

print("TARGET_COL:", TARGET_COL)
print("Task: STEMI vs NSTEMI (positive=STEMI)")
print(pd.Series(y).value_counts())
print("Positive rate:", float(np.mean(y)))
print("df filtered shape:", df.shape)


for k in features_used:
    features_used[k] = [f for f in features_used[k] if f != TARGET_COL]


TARGET_COL: typ_zawalu
Task: STEMI vs NSTEMI (positive=STEMI)
0    32
1    25
Name: count, dtype: int64
Positive rate: 0.43859649122807015
df filtered shape: (57, 117)


## NONNUMERIC_FEATURES

In [6]:
NONNUMERIC_FEATURES = set()
try:
    audit_df, path_audit = read_csv_required("audit_nonnumeric_features.csv")
    cand = ["feature","feature_name","column","variable","var","name"]
    col = next((c for c in cand if c in audit_df.columns), None)
    if col is None:
        col = audit_df.columns[0]
    NONNUMERIC_FEATURES = set(audit_df[col].astype(str).tolist())
    print("Loaded NONNUMERIC_FEATURES:", len(NONNUMERIC_FEATURES))
except FileNotFoundError:
    path_audit = None
    print("NOTE: audit_nonnumeric_features.csv not found (OK).")


NOTE: audit_nonnumeric_features.csv not found (OK).


## Matrix structure

In [7]:
YES_SET = {"yes","y","true","t","1","tak","on","present","positive","pos"}
NO_SET  = {"no","n","false","f","0","nie","off","absent","negative","neg"}

def _normalize_str_series(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.strip()
             .str.lower()
             .str.replace(",", ".", regex=False))

def _try_map_binary_object(col: pd.Series) -> tuple[pd.Series, bool]:
    norm = _normalize_str_series(col)
    uniq = sorted(norm.dropna().unique().tolist())
    if len(uniq) == 0:
        return col, False
    if len(set(uniq)) <= 2 and all(u in (YES_SET | NO_SET) for u in uniq):
        def map_one(v):
            if pd.isna(v): return np.nan
            vv = str(v).strip().lower()
            if vv in YES_SET: return 1
            if vv in NO_SET:  return 0
            return np.nan
        return norm.map(map_one).astype(float), True
    return col, False

def build_matrix(df_in: pd.DataFrame, feats: list[str], variant_name: str):
    feats = [f for f in feats if f in df_in.columns and f != TARGET_COL]
    feats = [f for f in feats if f not in NONNUMERIC_FEATURES]

    Xdf = df_in[feats].copy()

    recoded = 0
    for c in Xdf.columns:
        if Xdf[c].dtype == "object" or str(Xdf[c].dtype).startswith("bool"):
            mapped, ok = _try_map_binary_object(Xdf[c])
            if ok:
                Xdf[c] = mapped
                recoded += 1

    for c in Xdf.columns:
        if Xdf[c].dtype == "object":
            Xdf[c] = _normalize_str_series(Xdf[c])
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")

    all_nan = Xdf.columns[Xdf.isna().mean() == 1.0].tolist()
    constant = Xdf.columns[Xdf.nunique(dropna=True) <= 1].tolist()
    dropped = sorted(set(all_nan + constant))

    if dropped:
        pd.DataFrame({"feature": dropped}).to_csv(
            TAB_DIR / f"S8_{variant_name}_dropped_empty_or_constant.csv", index=False
        )
        Xdf = Xdf.drop(columns=dropped)

    diag = {
        "variant": variant_name,
        "n_governed_present": int(len(feats)),
        "n_recoded_binary": int(recoded),
        "n_dropped_empty_or_constant": int(len(dropped)),
        "n_used_matrix": int(Xdf.shape[1]),
    }
    return Xdf.values, np.array(Xdf.columns.tolist()), diag


## Wariantsa + diagnostics

In [8]:
variants_cfg = {}
diag_rows = []

for v in ["FULL", "CLINICAL", "BIOMARKERS"]:
    Xv, fn, diag = build_matrix(df, features_used[v], v)
    variants_cfg[v] = {"X": Xv, "features": fn}
    diag_rows.append(diag)

    pd.Series(list(fn)).to_csv(TAB_DIR / f"S8_{v}_features_used_final.csv", index=False, header=["feature"])
    print(f"{v}: X={Xv.shape}, n_features={len(fn)}")

diag_df = pd.DataFrame(diag_rows).sort_values("variant").reset_index(drop=True)
diag_df.to_csv(TAB_DIR / "S8_variant_matrix_diagnostics.csv", index=False)
print("Saved:", TAB_DIR / "S8_variant_matrix_diagnostics.csv")

try:
    display(diag_df)
except NameError:
    print(diag_df)

bad = diag_df[diag_df["n_used_matrix"] < 2]
if len(bad) > 0:
    raise RuntimeError("S8 cannot run: some variants have <2 usable predictors.\n" + bad.to_string(index=False))


summary = diag_df.copy()
summary["n_removed_total"] = summary["n_governed_present"] - summary["n_used_matrix"]
summary.to_csv(TAB_DIR / "S8_feature_reduction_summary.csv", index=False)
print("Saved:", TAB_DIR / "S8_feature_reduction_summary.csv")


FULL: X=(57, 53), n_features=53
CLINICAL: X=(57, 42), n_features=42
BIOMARKERS: X=(57, 10), n_features=10
Saved: /content/results/S8/tables/S8_variant_matrix_diagnostics.csv


,variant,n_governed_present,n_recoded_binary,n_dropped_empty_or_constant,n_used_matrix
0,BIOMARKERS,10,0,0,10
1,CLINICAL,58,20,16,42
2,FULL,69,20,16,53


Saved: /content/results/S8/tables/S8_feature_reduction_summary.csv


## Models + grids

In [9]:
REVISED_MODELS = {
    "LR_L1": LogisticRegression(penalty="l1", solver="saga", max_iter=8000, random_state=RANDOM_STATE),
    "LR_ELNET": LogisticRegression(penalty="elasticnet", solver="saga", max_iter=8000, random_state=RANDOM_STATE),
    "SVM_REG": SVC(probability=True, random_state=RANDOM_STATE),
    "RF_SHALLOW": RandomForestClassifier(random_state=RANDOM_STATE),
}

REVISED_GRIDS = {
    "LR_L1": {"clf__C": np.logspace(-3, 2, 10)},
    "LR_ELNET": {"clf__C": np.logspace(-3, 2, 10), "clf__l1_ratio": [0.1, 0.5, 0.9]},
    "SVM_REG": {"clf__C": np.logspace(-3, 2, 10), "clf__kernel": ["linear","rbf"], "clf__gamma": ["scale"]},
    "RF_SHALLOW": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [2,3,4,5],
        "clf__min_samples_leaf": [5,10,20],
        "clf__max_features": ["sqrt", 0.5],
    },
}

def make_pipeline(model_key: str, use_rfe: bool, n_features_to_select: int):
    pre_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_key in ["LR_L1","LR_ELNET","SVM_REG"]:
        pre_steps.append(("scaler", StandardScaler()))
    preprocess = Pipeline(pre_steps)

    steps = [("preprocess", preprocess)]
    if use_rfe:
        n_select = min(n_features_to_select,  max(2,  (n_features_to_select if n_features_to_select else 10)))
        rfe_est = LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)
        steps.append(("rfe", RFE(estimator=rfe_est, n_features_to_select=n_features_to_select)))
    steps.append(("clf", REVISED_MODELS[model_key]))
    return Pipeline(steps)


## Nested CV (OOF) + saving result tables

In [10]:
def nested_cv_oof(X: np.ndarray, y: np.ndarray, model_key: str, use_rfe: bool):
    oof = np.zeros(len(y), dtype=float)
    aucs, mccs, briers = [], [], []

    n_select_eff = None
    use_rfe_eff = False
    if use_rfe:
        n_select_eff = min(RFE_N_SELECT, X.shape[1]-1)
        use_rfe_eff = n_select_eff >= 2

    for fold_id, (itr, iva) in enumerate(cv_outer.split(X, y), start=1):
        pipe = make_pipeline(model_key, use_rfe=use_rfe_eff, n_features_to_select=(n_select_eff or RFE_N_SELECT))
        gs = GridSearchCV(pipe, REVISED_GRIDS[model_key], scoring="roc_auc", cv=cv_inner, n_jobs=-1, refit=True)
        gs.fit(X[itr], y[itr])

        probs = gs.best_estimator_.predict_proba(X[iva])[:, 1]
        oof[iva] = probs

        aucs.append(roc_auc_score(y[iva], probs))
        mccs.append(matthews_corrcoef(y[iva], (probs >= 0.5).astype(int)))
        briers.append(brier_score_loss(y[iva], probs))

    return {
        "auc_mean": float(np.mean(aucs)),
        "auc_sd": float(np.std(aucs, ddof=1)),
        "mcc_mean": float(np.mean(mccs)),
        "mcc_sd": float(np.std(mccs, ddof=1)),
        "brier_mean": float(np.mean(briers)),
        "brier_sd": float(np.std(briers, ddof=1)),
        "use_rfe_effective": bool(use_rfe_eff),
        "n_select_effective": int(n_select_eff) if use_rfe_eff else None
    }

rows = []
for variant_name in ["FULL","CLINICAL","BIOMARKERS"]:
    Xv = variants_cfg[variant_name]["X"]
    print("\nS8 nested CV:", variant_name, "| X:", Xv.shape)

    for model_key in REVISED_MODELS.keys():
        info = nested_cv_oof(Xv, y, model_key=model_key, use_rfe=USE_RFE)
        rows.append({
            "variant": variant_name,
            "model": model_key + ("_RFE" if info["use_rfe_effective"] else ""),
            **info,
            "n_samples": int(len(y)),
            "n_features": int(Xv.shape[1]),
        })

df_nested = pd.DataFrame(rows).sort_values(["variant","auc_mean"], ascending=[True, False]).reset_index(drop=True)
df_nested.to_csv(TAB_DIR / "S8_nestedcv_results_revised.csv", index=False)
print("Saved:", TAB_DIR / "S8_nestedcv_results_revised.csv")

try:
    display(df_nested.head(12))
except NameError:
    print(df_nested.head(12))



S8 nested CV: FULL | X: (57, 53)

S8 nested CV: CLINICAL | X: (57, 42)

S8 nested CV: BIOMARKERS | X: (57, 10)
Saved: /content/results/S8/tables/S8_nestedcv_results_revised.csv


,variant,model,auc_mean,auc_sd,mcc_mean,mcc_sd,brier_mean,brier_sd,use_rfe_effective,n_select_effective,n_samples,n_features
0,BIOMARKERS,RF_SHALLOW,0.916190,0.076384,0.739048,0.151388,0.136653,0.021058,False,None,57,10
1,BIOMARKERS,LR_L1,0.849524,0.088743,0.598756,0.250842,0.157773,0.078527,False,None,57,10
2,BIOMARKERS,LR_ELNET,0.820952,0.036359,0.318481,0.348178,0.185280,0.058049,False,None,57,10
3,BIOMARKERS,SVM_REG,0.812381,0.114415,0.550369,0.269534,0.173688,0.071435,False,None,57,10
4,CLINICAL,RF_SHALLOW,0.612381,0.147450,0.198326,0.317574,0.240378,0.014295,False,None,57,42
5,CLINICAL,SVM_REG,0.532381,0.114107,-0.033113,0.266930,0.253126,0.013886,False,None,57,42
6,CLINICAL,LR_ELNET,0.505714,0.061112,0.127171,0.164324,0.302268,0.111108,False,None,57,42
7,CLINICAL,LR_L1,0.479524,0.090783,0.005714,0.050440,0.344195,0.111967,False,None,57,42
8,FULL,LR_L1,1.000000,0.000000,0.969031,0.069249,0.063489,0.025363,False,None,57,53
9,FULL,LR_ELNET,1.000000,0.000000,0.105830,0.236643,0.211926,0.006591,False,None,57,53


## Winner per variant + run_log

In [11]:
winners = []
for v in ["FULL","CLINICAL","BIOMARKERS"]:
    sub = df_nested[df_nested["variant"] == v].copy()
    best = sub.sort_values("auc_mean", ascending=False).iloc[0]
    winners.append(best)

df_winners = pd.DataFrame(winners).reset_index(drop=True)
df_winners.to_csv(TAB_DIR / "S8_revised_winner_by_variant.csv", index=False)
print("Saved:", TAB_DIR / "S8_revised_winner_by_variant.csv")

run_log = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "random_state": RANDOM_STATE,
    "target": {
        "TARGET_COL": TARGET_COL,
        "task": "STEMI vs NSTEMI (positive=STEMI)",
        "n_samples_after_filtering": int(len(y)),
        "positive_rate": float(np.mean(y)),
    },
    "cv": {"outer": N_OUTER, "inner": N_INNER},
    "rfe": {"enabled": USE_RFE, "n_select_default": RFE_N_SELECT},
    "inputs": {
        "processed_dataset.csv": str(path_processed),
        "features_used_FULL.csv": str(p_full),
        "features_used_CLINICAL.csv": str(p_clin),
        "features_used_BIOMARKERS.csv": str(p_bio),
        "audit_nonnumeric_features.csv": str(path_audit) if path_audit else None,
    },
    "outputs": {
        "S8_variant_matrix_diagnostics.csv": str(TAB_DIR / "S8_variant_matrix_diagnostics.csv"),
        "S8_feature_reduction_summary.csv": str(TAB_DIR / "S8_feature_reduction_summary.csv"),
        "S8_nestedcv_results_revised.csv": str(TAB_DIR / "S8_nestedcv_results_revised.csv"),
        "S8_revised_winner_by_variant.csv": str(TAB_DIR / "S8_revised_winner_by_variant.csv"),
    },
    "notes": [
        "S8 fixed: binary recoding (YES/NO/TAK/NIE) performed before numeric coercion (S9-consistent).",
        "Dropped features are those that are all-NaN or constant after coercion within the filtered dataset.",
        "Nested CV used for unbiased model selection and performance estimation (internal validation)."
    ]
}

log_path = LOG_DIR / "S8_run_log.json"
log_path.write_text(json.dumps(run_log, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", log_path)


Saved: /content/results/S8/tables/S8_revised_winner_by_variant.csv
Saved: /content/results/S8/logs/S8_run_log.json


/tmp/ipython-input-3448658472.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",
